# Практика 3. Соединение таблиц, группировка и сводные таблицы

Лекция 2: Продвинутые методы библиотеки pandas для анализа данных.

**Цель практики.** Датасет о продажах пиццы за год разбит на четыре связанные
таблицы (заказы, позиции заказов, пиццы, типы пицц) — типичная реляционная
структура. Нужно собрать из неё одну аналитическую таблицу, посчитать выручку
и статистики по категориям и времени, и научиться переключаться между
широким (wide) и длинным (long, tidy) представлением данных.

**Учебные цели:**
- соединять несколько таблиц через `merge` по составным цепочкам ключей;
- понимать разницу между `concat` (сшивание по строкам/столбцам) и `merge`/`join` (соединение по ключу);
- строить групповые агрегаты и работать с `MultiIndex` (`.loc`, `unstack`/`stack`);
- строить сводные таблицы `pivot_table` и переводить их в tidy-формат через `melt`;
- считать оконные статистики: скользящее среднее (`rolling`) и накопительную сумму (`cumsum`).

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 20)
plt.rcParams['figure.figsize'] = (9, 4)

# DATA_DIR = '../datasets'

pizzas = pd.read_csv('pizzas.csv')
pizza_types = pd.read_csv('pizza_types.csv', encoding='latin-1')
orders = pd.read_csv('orders.csv')
order_details = pd.read_csv('order_details.csv')

for name, df in [('pizzas', pizzas), ('pizza_types', pizza_types),
                  ('orders', orders), ('order_details', order_details)]:
    print(f'{name}: {df.shape}')

order_details.head()

pizzas: (96, 4)
pizza_types: (32, 4)
orders: (21350, 3)
order_details: (48620, 4)


,order_details_id,order_id,pizza_id,quantity
0,1,1,hawaiian_m,1
1,2,2,classic_dlx_m,1
2,3,2,five_cheese_l,1
3,4,2,ital_supr_l,1
4,5,2,mexicana_m,1


### Задание 1


1. Посмотрите на типы столбцов всех четырёх таблиц (`dtypes`) и найдите столбцы,
   по которым таблицы связаны друг с другом (`order_id`, `pizza_id`, `pizza_type_id`).
2. Соедините таблицы последовательно через `merge` в одну широкую таблицу `orders_full`:
   `order_details` → `orders` (по `order_id`) → `pizzas` (по `pizza_id`) → `pizza_types`
   (по `pizza_type_id`). Используйте `how='left'`, так как `order_details` — главная таблица фактов.
3. Проверьте, что число строк не изменилось по сравнению с `order_details`, и что
   после соединения не появилось пропусков (`isna().sum()`).

In [4]:
# ваш код здесь
# подсказка: выведите dtypes для каждой из 4 таблиц и определите столбцы-ключи
# подсказка: ключи: order_details.order_id -> orders.order_id, order_details.pizza_id -> pizzas.pizza_id,
# подсказка: pizzas.pizza_type_id -> pizza_types.pizza_type_id
pizzas.info()

<class 'pandas.DataFrame'>
RangeIndex: 96 entries, 0 to 95
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   pizza_id       96 non-null     str    
 1   pizza_type_id  96 non-null     str    
 2   size           96 non-null     str    
 3   price          96 non-null     float64
dtypes: float64(1), str(3)
memory usage: 3.1 KB


In [5]:
pizza_types.info()

<class 'pandas.DataFrame'>
RangeIndex: 32 entries, 0 to 31
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   pizza_type_id  32 non-null     str  
 1   name           32 non-null     str  
 2   category       32 non-null     str  
 3   ingredients    32 non-null     str  
dtypes: str(4)
memory usage: 1.1 KB


In [6]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 21350 entries, 0 to 21349
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   order_id  21350 non-null  int64
 1   date      21350 non-null  str  
 2   time      21350 non-null  str  
dtypes: int64(1), str(2)
memory usage: 500.5 KB


In [7]:
order_details.info()

<class 'pandas.DataFrame'>
RangeIndex: 48620 entries, 0 to 48619
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   order_details_id  48620 non-null  int64
 1   order_id          48620 non-null  int64
 2   pizza_id          48620 non-null  str  
 3   quantity          48620 non-null  int64
dtypes: int64(3), str(1)
memory usage: 1.5 MB


In [11]:
## пункт 2

orders_full = (
    order_details
    .merge(orders, on='order_id', how='left')
    .merge(pizzas, on='pizza_id', how='left')
    .merge(pizza_types, on='pizza_type_id', how='left')
)


In [13]:
order_details.info()

<class 'pandas.DataFrame'>
RangeIndex: 48620 entries, 0 to 48619
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   order_details_id  48620 non-null  int64
 1   order_id          48620 non-null  int64
 2   pizza_id          48620 non-null  str  
 3   quantity          48620 non-null  int64
dtypes: int64(3), str(1)
memory usage: 1.5 MB


In [14]:
orders_full.info()

<class 'pandas.DataFrame'>
RangeIndex: 48620 entries, 0 to 48619
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   order_details_id  48620 non-null  int64  
 1   order_id          48620 non-null  int64  
 2   pizza_id          48620 non-null  str    
 3   quantity          48620 non-null  int64  
 4   date              48620 non-null  str    
 5   time              48620 non-null  str    
 6   pizza_type_id     48620 non-null  str    
 7   size              48620 non-null  str    
 8   price             48620 non-null  float64
 9   name              48620 non-null  str    
 10  category          48620 non-null  str    
 11  ingredients       48620 non-null  str    
dtypes: float64(1), int64(3), str(8)
memory usage: 4.5 MB


In [15]:
orders_full.isna().sum()

order_details_id    0
order_id            0
pizza_id            0
quantity            0
date                0
time                0
pizza_type_id       0
size                0
price               0
name                0
category            0
ingredients         0
dtype: int64

### Задание 2


`concat` сшивает объекты по строкам или по столбцам **без** сопоставления по ключу
(нужно только совпадение структуры), а `merge`/`join` соединяют таблицы **по значению
ключевого столбца**, добавляя новые столбцы. Продемонстрируйте это на практике:

1. Заново соберите `orders_full` (как в задании 1) и превратите столбец `date` в `datetime`.
2. Разбейте `orders_full` на две части по медианной дате: `part_before` (дата ≤ медианы)
   и `part_after` (дата > медианы).
3. Склейте части обратно через `pd.concat([...], axis=0)`, отсортируйте по индексу
   и убедитесь, что результат совпадает с исходной таблицей (`.equals()`).

In [ ]:
# ваш код здесь
# подсказка: пересоберите orders_full (см. задание 1) и приведите date к datetime через pd.to_datetime
# подсказка: найдите median_date = orders_full['date'].median()
# подсказка: разбейте на part_before (date <= median_date) и part_after (date > median_date)

In [17]:
orders['date'] = pd.to_datetime(orders['date'])

In [19]:
orders_full = (
    order_details
    .merge(orders, on='order_id', how='left')
    .merge(pizzas, on='pizza_id', how='left')
    .merge(pizza_types, on='pizza_type_id', how='left')
)


In [21]:
median_date = orders_full['date'].median()

In [22]:
part_before = orders_full[orders_full['date'] <= median_date]
part_after = orders_full[orders_full['date'] > median_date]

In [23]:
# ваш код здесь
# подсказка: склейте part_before и part_after через pd.concat([...], axis=0), отсортируйте по индексу
# подсказка: сравните с orders_full.sort_index() через .equals()
new_orders_full =pd.concat([part_before, part_after], axis=0).sort_index()

In [24]:
new_orders_full.equals(orders_full.sort_index())

True

### Задание 3


1. Заново соберите `orders_full`.
2. Добавьте столбец `revenue = quantity * price`.
3. Распарсите `date` в `datetime`, а `date` + `time` объедините в единый столбец
   `order_datetime` типа `datetime64`.

In [25]:
# ваш код здесь
# подсказка: пересоберите orders_full (см. задание 1)
# подсказка: добавьте revenue = quantity * price
# подсказка: приведите date к datetime, затем соберите order_datetime = pd.to_datetime(date.dt.strftime(...) + ' ' + time)
orders_full = (
    order_details
    .merge(orders, on='order_id', how='left')
    .merge(pizzas, on='pizza_id', how='left')
    .merge(pizza_types, on='pizza_type_id', how='left')
)

In [28]:
orders_full['revenue'] = orders_full['quantity'] * orders_full['price']

In [30]:
orders_full['date'] = pd.to_datetime(orders_full['date'])
orders_full['order_datetime'] = pd.to_datetime(
    orders_full['date'].astype(str) + ' ' + orders_full['time'].astype(str)
)

orders_full[['date', 'time', 'order_datetime']].head()

,date,time,order_datetime
0,2015-01-01,11:38:36,2015-01-01 11:38:36
1,2015-01-01,11:57:40,2015-01-01 11:57:40
2,2015-01-01,11:57:40,2015-01-01 11:57:40
3,2015-01-01,11:57:40,2015-01-01 11:57:40
4,2015-01-01,11:57:40,2015-01-01 11:57:40


### Задание 4


1. Заново соберите `orders_full` и добавьте `revenue` (как в задании 3).
2. Посчитайте `groupby('category')['revenue'].agg(['sum', 'mean', 'count'])`.
3. Посчитайте группировку по двум ключам `['category', 'size']` — получится таблица
   с `MultiIndex`. Обратитесь к конкретной комбинации через `.loc[('Classic', 'L')]`.
4. Разверните таблицу в широкий формат через `.unstack('size')`, а затем верните
   обратно через `.stack()`.

In [ ]:
# ваш код здесь
# подсказка: пересоберите orders_full и revenue (см. задания 1 и 3)
# подсказка: cat_stats = orders_full.groupby('category')['revenue'].agg(['sum', 'mean', 'count'])

In [31]:
orders_full = (
    order_details
    .merge(orders, on='order_id', how='left')
    .merge(pizzas, on='pizza_id', how='left')
    .merge(pizza_types, on='pizza_type_id', how='left')
)
orders_full['revenue'] = orders_full['quantity'] * orders_full['price']

In [ ]:
# ваш код здесь
# подсказка: cat_size_stats = orders_full.groupby(['category', 'size'])['revenue'].agg(['sum', 'mean', 'count'])
# подсказка: убедитесь, что cat_size_stats.index — это MultiIndex
cat_size_stats = orders_full.groupby(['category', 'size'])['revenue'].agg(['sum', 'mean', 'count'])
cat_size_stats.index 

MultiIndex([('Chicken',   'L'),
            ('Chicken',   'M'),
            ('Chicken',   'S'),
            ('Classic',   'L'),
            ('Classic',   'M'),
            ('Classic',   'S'),
            ('Classic',  'XL'),
            ('Classic', 'XXL'),
            ('Supreme',   'L'),
            ('Supreme',   'M'),
            ('Supreme',   'S'),
            ( 'Veggie',   'L'),
            ( 'Veggie',   'M'),
            ( 'Veggie',   'S')],
           names=['category', 'size'])

In [40]:
cat_size_stats.loc[('Classic', 'L')]

sum      74518.500000
mean        18.690369
count     3987.000000
Name: (Classic, L), dtype: float64

In [ ]:
# ваш код здесь
# подсказка: cat_size_wide = cat_size_stats['sum'].unstack('size')

In [ ]:
# ваш код здесь
# подсказка: cat_size_long = cat_size_wide.stack()

### Задание 5


1. Заново соберите `orders_full`, добавьте `revenue`, приведите `date` к `datetime`
   и добавьте столбец `month` (номер месяца).
2. Постройте сводную таблицу `pivot_table`: строки — `month`, столбцы — `category`,
   значения — сумма `revenue`.

In [ ]:
# ваш код здесь
# подсказка: пересоберите orders_full, revenue, date (см. предыдущие задания), добавьте month = date.dt.month
# подсказка: pivot_month_category = orders_full.pivot_table(index='month', columns='category', values='revenue', aggfunc='sum')

### Задание 6


1. Постройте `pivot_month_category`, как в задании 5.
2. Превратите её обратно в длинный (tidy) формат: одна строка = одна комбинация
   (месяц, категория, выручка). Используйте `reset_index()` + `melt(id_vars='month', ...)`.

In [ ]:
# ваш код здесь
# подсказка: постройте pivot_month_category (см. задание 5)
# подсказка: tidy_month_category = pivot_month_category.reset_index().melt(id_vars='month', var_name='category', value_name='revenue')

### Задание 7


1. Заново соберите `orders_full` с `revenue` и `date`, посчитайте дневную выручку
   `daily_revenue = orders_full.groupby('date')['revenue'].sum()`, отсортированную по дате.
2. Посчитайте скользящее среднее за 7 дней (`rolling(window=7).mean()`) и постройте график
   дневной выручки вместе со сглаженной линией.
3. Посчитайте накопительную сумму выручки **по месяцам** (внутри каждого месяца
   накопление начинается заново) через `groupby('month')['revenue'].cumsum()`.

In [ ]:
# ваш код здесь
# подсказка: пересоберите orders_full и revenue (см. задание 3), приведите date к datetime
# подсказка: daily_revenue = orders_full.groupby('date')['revenue'].sum().sort_index()
# подсказка: rolling_7d = daily_revenue.rolling(window=7).mean()
# подсказка: постройте оба ряда на одном графике

In [ ]:
# ваш код здесь
# подсказка: daily_df = daily_revenue.reset_index(); добавьте month = date.dt.month
# подсказка: daily_df['cum_revenue_in_month'] = daily_df.groupby('month')['revenue'].cumsum()